In [10]:
from torch.distributed.checkpoint.format_utils import dcp_to_torch_save
import torch

CKPT_DIR = "/ccn2/u/khaiaw/Code/baselines/dinov3/babyview/outputs/grad_accum_1/ckpt/119999"
OUT_PT   = f"{CKPT_DIR}/model.pt"

dcp_to_torch_save(CKPT_DIR, OUT_PT)   # run on a single process (CPU) :contentReference[oaicite:3]{index=3}

sd = torch.load(OUT_PT, map_location="cpu", weights_only=False)
print("num keys:", len(sd))
print("first keys:", list(sd.keys())[:30])
print(sd['model'].keys())

num keys: 3
first keys: ['iteration', 'model', 'optimizer']
dict_keys(['student.backbone.cls_token', 'student.backbone.mask_token', 'student.backbone.patch_embed.proj.weight', 'student.backbone.patch_embed.proj.bias', 'student.backbone.blocks.0.norm1.weight', 'student.backbone.blocks.0.norm1.bias', 'student.backbone.blocks.0.attn.qkv.weight', 'student.backbone.blocks.0.attn.qkv.bias', 'student.backbone.blocks.0.attn.proj.weight', 'student.backbone.blocks.0.attn.proj.bias', 'student.backbone.blocks.0.ls1.gamma', 'student.backbone.blocks.0.norm2.weight', 'student.backbone.blocks.0.norm2.bias', 'student.backbone.blocks.0.mlp.fc1.weight', 'student.backbone.blocks.0.mlp.fc1.bias', 'student.backbone.blocks.0.mlp.fc2.weight', 'student.backbone.blocks.0.mlp.fc2.bias', 'student.backbone.blocks.0.ls2.gamma', 'student.backbone.blocks.1.norm1.weight', 'student.backbone.blocks.1.norm1.bias', 'student.backbone.blocks.1.attn.qkv.weight', 'student.backbone.blocks.1.attn.qkv.bias', 'student.backbone.bl

In [11]:
import re
import torch
from transformers import DINOv3ViTConfig, DINOv3ViTModel

# sd = torch.load(...);  # you already did this
ckpt = sd["model"]

# Choose which backbone to export (recommend teacher or model_ema)
PREFIX = "teacher.backbone."      # or "model_ema.backbone."
src = {k[len(PREFIX):]: v.cpu() for k, v in ckpt.items() if k.startswith(PREFIX)}

# ---- Infer core dims from tensors ----
hidden = src["cls_token"].shape[-1]
patch = src["patch_embed.proj.weight"].shape[-1]
in_ch = src["patch_embed.proj.weight"].shape[1]
intermediate = src["blocks.0.mlp.fc1.weight"].shape[0]

block_ids = sorted({int(m.group(1)) for k in src for m in [re.match(r"blocks\.(\d+)\.", k)] if m})
num_layers = (max(block_ids) + 1) if block_ids else 0

# Common ViT convention: head_dim=64 => num_heads = hidden/64
num_heads = hidden // 64  # adjust if you trained with a different head_dim

config = DINOv3ViTConfig(
    image_size=224,                # doesn’t have to match training exactly; model supports varying sizes via RoPE
    patch_size=patch,
    num_channels=in_ch,
    hidden_size=hidden,
    intermediate_size=intermediate,
    num_hidden_layers=num_layers,
    num_attention_heads=num_heads,
    num_register_tokens=0,          # your ckpt has no register_tokens key
    query_bias=True, key_bias=True, value_bias=True, proj_bias=True,
    use_gated_mlp=False,            # your ckpt has fc1/fc2 (maps to up_proj/down_proj)
)

model = DINOv3ViTModel(config)

hf = {}

# ---- Embeddings ----
def tok3(x: torch.Tensor) -> torch.Tensor:
    # ckpt: (1, H)  -> HF: (1, 1, H)
    return x.unsqueeze(1) if x.ndim == 2 else x

hf["embeddings.cls_token"]  = tok3(src["cls_token"])
hf["embeddings.mask_token"] = tok3(src["mask_token"])

hf["embeddings.patch_embeddings.weight"] = src["patch_embed.proj.weight"]
hf["embeddings.patch_embeddings.bias"] = src["patch_embed.proj.bias"]

# ---- Transformer blocks ----
H = hidden
for i in range(num_layers):
    hf[f"layer.{i}.norm1.weight"] = src[f"blocks.{i}.norm1.weight"]
    hf[f"layer.{i}.norm1.bias"]   = src[f"blocks.{i}.norm1.bias"]

    # qkv -> q_proj/k_proj/v_proj (split along output dim)
    qkv_w = src[f"blocks.{i}.attn.qkv.weight"]
    qkv_b = src.get(f"blocks.{i}.attn.qkv.bias", None)

    hf[f"layer.{i}.attention.q_proj.weight"] = qkv_w[0:H, :]
    hf[f"layer.{i}.attention.k_proj.weight"] = qkv_w[H:2*H, :]
    hf[f"layer.{i}.attention.v_proj.weight"] = qkv_w[2*H:3*H, :]

    if qkv_b is not None:
        hf[f"layer.{i}.attention.q_proj.bias"] = qkv_b[0:H]
        hf[f"layer.{i}.attention.k_proj.bias"] = qkv_b[H:2*H]
        hf[f"layer.{i}.attention.v_proj.bias"] = qkv_b[2*H:3*H]

    # proj -> o_proj
    hf[f"layer.{i}.attention.o_proj.weight"] = src[f"blocks.{i}.attn.proj.weight"]
    hf[f"layer.{i}.attention.o_proj.bias"]   = src[f"blocks.{i}.attn.proj.bias"]

    # LayerScale: timm uses gamma; HF uses lambda1
    hf[f"layer.{i}.layer_scale1.lambda1"] = src[f"blocks.{i}.ls1.gamma"]

    hf[f"layer.{i}.norm2.weight"] = src[f"blocks.{i}.norm2.weight"]
    hf[f"layer.{i}.norm2.bias"]   = src[f"blocks.{i}.norm2.bias"]

    # MLP: fc1/fc2 -> up_proj/down_proj (ArceeMLP)
    hf[f"layer.{i}.mlp.up_proj.weight"]   = src[f"blocks.{i}.mlp.fc1.weight"]
    hf[f"layer.{i}.mlp.up_proj.bias"]     = src[f"blocks.{i}.mlp.fc1.bias"]
    hf[f"layer.{i}.mlp.down_proj.weight"] = src[f"blocks.{i}.mlp.fc2.weight"]
    hf[f"layer.{i}.mlp.down_proj.bias"]   = src[f"blocks.{i}.mlp.fc2.bias"]

    hf[f"layer.{i}.layer_scale2.lambda1"] = src[f"blocks.{i}.ls2.gamma"]

# Final norm
hf["norm.weight"] = src["norm.weight"]
hf["norm.bias"]   = src["norm.bias"]

missing, unexpected = model.load_state_dict(hf, strict=False)
print("missing (ok if small):", missing[:20], "…", len(missing))
print("unexpected:", unexpected[:20], "…", len(unexpected))

OUT_DIR = f"{CKPT_DIR}/huggingface"
model.save_pretrained(OUT_DIR, safe_serialization=True)
print("saved to", OUT_DIR)


missing (ok if small): ['embeddings.register_tokens'] … 1
unexpected: [] … 0
saved to /ccn2/u/khaiaw/Code/baselines/dinov3/babyview/outputs/grad_accum_1/ckpt/119999/huggingface
